# Test OpenRouter Models for Trip Generation

This notebook helps you test which free models work best with your OpenRouter key for trip planning.

In [1]:
import os
import sys
import time
from datetime import datetime
from dotenv import load_dotenv
from openai import OpenAI

# Add src to path
sys.path.append(os.path.join(os.getcwd(), 'src'))

load_dotenv()

api_key = os.getenv('OPENROUTER_API_KEY')
print(f"✅ API Key loaded: {'Yes' if api_key else 'No'}")
if api_key:
    print(f"Key starts with: {api_key[:10]}...")

✅ API Key loaded: Yes
Key starts with: sk-or-v1-7...


In [3]:
from openai import OpenAI

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key,
    default_headers={
        "HTTP-Referer": "https://egypt-trip-planner.com",
        "X-Title": "Trip Planner Tester"
    }
)

print("✅ Client initialized with custom headers")

✅ Client initialized with custom headers


In [5]:
# ====================== Test Configuration ======================
from datetime import datetime

test_config = {
    "destinations": ["Cairo & Giza"],
    "budget": "1500 EGP",
    "group_size": 2,
    "start_date": datetime(2026, 4, 15),
    "end_date": datetime(2026, 4, 20),
    "travel_styles": ["Historical", "Food & Dining"],
    "food_preferences": "Vegetarian",
    "trip_pace": "Moderate",
    "must_visit": "Pyramids of Giza"
}

# Create the test prompt
test_prompt = f"""Create a {(test_config['end_date'] - test_config['start_date']).days}-day trip plan for {', '.join(test_config['destinations'])}.

Budget: {test_config['budget']} for {test_config['group_size']} people
Must Visit: {test_config['must_visit']}
Travel Style: {', '.join(test_config['travel_styles'])}
Food Preference: {test_config['food_preferences']}
Pace: {test_config['trip_pace']}

Provide a clear day-by-day itinerary, overview, and detailed budget breakdown."""

print("✅ Test prompt created successfully")
print(f"Prompt length: {len(test_prompt)} characters")

✅ Test prompt created successfully
Prompt length: 265 characters


In [ ]:
# Models to Test (Best Free Models - April 2026)
models_to_test = [
    "openrouter/free",                          # Auto best free model
    "stepfun/step-3.5-flash:free",
    "nvidia/nemotron-3-super-120b-a12b:free",
    "arcee-ai/trinity-large-preview:free",
    "nvidia/nemotron-3-super-120b-a12b:free"
]

results = []

for model in models_to_test:
    print(f"\n{'='*70}")
    print(f"Testing: {model}")
    print(f"{'='*70}")
    
    try:
        start = time.time()
        
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": "You are a professional Egypt travel planner. Be practical, respect budget, and follow user requirements strictly."},
                {"role": "user", "content": test_prompt}
            ],
            temperature=0.7,
            max_tokens=1200
        )
        
        duration = time.time() - start
        content = response.choices[0].message.content
        usage = response.usage
        
        print(f"✅ SUCCESS")
        print(f"   Time     : {duration:.2f} seconds")
        print(f"   Tokens   : {usage.total_tokens} (in: {usage.prompt_tokens}, out: {usage.completion_tokens})")
        print(f"   Preview  : {content[:180]}...")
        
        results.append({
            "model": model,
            "status": "SUCCESS",
            "time": round(duration, 2),
            "tokens": usage.total_tokens,
            "preview": content[:180]
        })
        
    except Exception as e:
        error_str = str(e)[:150]
        print(f"❌ FAILED - {error_str}")
        results.append({"model": model, "status": "FAILED", "error": error_str})
    
    time.sleep(1.5)  # Small delay to reduce rate limit pressure


Testing: openrouter/free
✅ SUCCESS
   Time     : 6.64 seconds
   Tokens   : 1298 (in: 98, out: 1200)
   Preview  : 

Here’s a **practical 5-day Cairo & Giza itinerary** for 2 people, strictly adhering to your budget (1500 EGP), vegetarian preferences, and historical/food focus. This plan priori...

Testing: stepfun/step-3.5-flash:free
✅ SUCCESS
   Time     : 11.97 seconds
   Tokens   : 1307 (in: 107, out: 1200)
   Preview  : Excellent! As your Egypt travel planner, I’ve crafted a realistic, budget-conscious, and enriching 5-day itinerary for Cairo & Giza that hits all your requirements: historical focu...

Testing: nvidia/nemotron-3-super-120b-a12b:free
✅ SUCCESS
   Time     : 37.12 seconds
   Tokens   : 1312 (in: 112, out: 1200)
   Preview  : We need to produce 5-day plan for Cairo & Giza, budget 1500 EGP for 2 people, historical, food & dining, vegetarian, moderate pace. Must include overview and detailed budget breakd...

Testing: arcee-ai/trinity-large-preview:free
✅ SUCCESS
   Ti

In [7]:
# Final Summary
print(f"\n{'='*70}")
print("FINAL TEST SUMMARY")
print(f"{'='*70}")

for r in results:
    if r["status"] == "SUCCESS":
        print(f"✅ {r['model']:45} | {r['time']:5}s | {r['tokens']} tokens")
    else:
        print(f"❌ {r['model']:45} | FAILED → {r['error']}")


FINAL TEST SUMMARY
✅ openrouter/free                               |  6.64s | 1298 tokens
✅ stepfun/step-3.5-flash:free                   | 11.97s | 1307 tokens
✅ nvidia/nemotron-3-super-120b-a12b:free        | 37.12s | 1312 tokens
✅ arcee-ai/trinity-large-preview:free           | 85.42s | 629 tokens
✅ google/gemma-4-31b-it:free                    | 30.37s | 1155 tokens
